# 06 Attention 与 Transformer 基础

## 本节目标

- 理解 Query、Key、Value 的基本作用。
- 手写 scaled dot-product attention。
- 可视化注意力权重。
- 用一个小 Transformer Block 观察输入输出形状。

## 背景问题

本节关注的问题是：模型在处理序列中的某个位置时，如何直接参考其他位置的信息？

Attention 通过相关性权重汇总信息。这个概念可以从代码角度理解为：先计算位置之间的匹配分数，再用 softmax 得到权重，最后对 Value 加权求和。

## 核心概念

- Query：当前位置发出的查询。
- Key：每个位置用于和 Query 匹配的表示。
- Value：被加权汇总的信息。
- Attention weights：每个位置对其他位置的关注比例。
- Transformer Block：通常包含注意力、残差连接、归一化和前馈网络。

## 最小代码示例

下面先构造一个很小的序列张量。这个实验用于观察注意力权重矩阵的形状和含义。

In [ ]:
import torch
from torch import nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

# batch=1, seq_len=4, dim=3
x = torch.tensor([[[1.0, 0.0, 0.5],
                   [0.8, 0.2, 0.4],
                   [0.0, 1.0, 0.3],
                   [0.1, 0.9, 0.2]]])
print("input shape:", x.shape)

## 完整实验

这里的关键不是公式本身，而是看清 `QK^T -> softmax -> 加权 V` 这条计算链路。scaled dot-product attention 可以理解为：先计算 Query 和 Key 的相似度，再按权重汇总 Value。

### 手写 Attention

下面直接令 `Q=K=V=x`，也就是自注意力。这样可以先关注权重矩阵，而不是被额外的线性投影分散注意力。

In [ ]:
def scaled_dot_product_attention(q, k, v):
    d_k = q.size(-1)
    scores = q @ k.transpose(-2, -1) / np.sqrt(d_k)
    weights = torch.softmax(scores, dim=-1)
    output = weights @ v
    return output, weights

attn_output, attn_weights = scaled_dot_product_attention(x, x, x)
print("attention output shape:", attn_output.shape)
print("attention weights:")
print(attn_weights[0])

In [ ]:
plt.figure(figsize=(4, 3.5))
plt.imshow(attn_weights[0].detach().numpy(), cmap="viridis")
plt.colorbar(label="weight")
plt.xlabel("key position")
plt.ylabel("query position")
plt.title("Attention weights")
plt.tight_layout()
plt.show()

### 小 Transformer Block

下面使用 `nn.MultiheadAttention` 构造一个很小的 Transformer Block。重点观察输入和输出 shape 是否一致，这决定了 block 能否继续堆叠。

In [ ]:
class TinyTransformerBlock(nn.Module):
    def __init__(self, embed_dim=8, num_heads=2, ff_dim=16):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, embed_dim),
        )
        self.norm2 = nn.LayerNorm(embed_dim)

    def forward(self, x):
        attn_out, weights = self.attn(x, x, x, need_weights=True)
        x = self.norm1(x + attn_out)
        ffn_out = self.ffn(x)
        x = self.norm2(x + ffn_out)
        return x, weights

block = TinyTransformerBlock(embed_dim=8, num_heads=2)
toy_tokens = torch.randn(2, 5, 8)
out, weights = block(toy_tokens)
print("input shape:", toy_tokens.shape)
print("output shape:", out.shape)
print("weights shape:", weights.shape)

## 实验观察

从运行结果可以看到，注意力权重矩阵的每一行都接近一个概率分布，表示一个 query 位置如何分配对 key 位置的关注。小 Transformer Block 的输出 shape 和输入一致，这让多个 block 可以继续堆叠。

## 常见错误

- softmax 维度写错，导致权重没有在 key 维度归一化。
- `embed_dim` 不能被 `num_heads` 整除。
- 混淆 batch-first 和 seq-first 的输入格式。
- 忘记残差连接前后的张量形状必须一致。

调试时可以优先检查 attention scores、weights 和输出张量的 shape。

In [ ]:
row_sums = attn_weights[0].sum(dim=-1)
print("each attention row sums to:", row_sums)

## 概念问答

**Q：Attention 权重能完全解释模型吗？**  
A：不能简单等同于完整解释。它能提供一个观察角度，但模型结果还会受到投影层、前馈层和训练目标影响。

**Q：为什么 Transformer 可以堆叠很多层？**  
A：每个 block 通常保持输入输出维度一致，并使用残差连接和归一化帮助训练稳定。

**Q：为什么要除以 `sqrt(d_k)`？**  
A：当维度较大时，点积可能变得很大，缩放能让 softmax 更稳定。

## 小结

Attention 是基于相关性的加权汇总机制。理解 Q/K/V、权重矩阵和输出形状，是继续学习 Transformer 的关键起点。